In [1]:
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import cv2, os, shutil, glob, json
import numpy as np
import pandas as pd
import torch.nn as nn
import seaborn as sns
from pathlib import Path
pd.set_option('display.max_columns', None)

In [2]:
OPTIONS = json.loads(open('../../../Task/info.json', 'r').read())
OPTIONS

{'img_size': [16, 256, 256],
 'step': 3,
 'model': 'segresnet',
 'lr': 0.0001,
 'loss': 'dice_focal',
 'batch': 2,
 'scheduler': 'plateau'}

In [3]:
IMG_SIZE = tuple(OPTIONS.get('img_size'))
IMG_SIZE

(16, 256, 256)

In [4]:
def loadFile(path, size=(128, 128, 128)):
    file = Path(path)
    ext  = file.suffix.lower()

    if ext == '.png':
        return cv2.imread(path)
    
    if ext == '.npy':
        return np.load(path)
    
    if ext == '.dat':
        return np.reshape(np.fromfile(path, dtype=np.single), size)

    return None

def getFiles(path, limit=None, shuffle=False):
    target = sorted(glob.glob(os.path.join(path, '*')))
    if shuffle:
        np.random.shuffle(target) 
    return target[:limit]

In [5]:
images = [loadFile(img, IMG_SIZE) for img in getFiles('images')]
masks  = [loadFile(img, IMG_SIZE) for img in getFiles('masks')]

IMG_SIZE = images[0].shape[0]
print(len(images), IMG_SIZE)
images[:5]

1782 16


[array([[[0.5026313 , 0.5026313 , 0.5026313 , ..., 0.5669865 ,
          0.5692591 , 0.5694053 ],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.5793223 ,
          0.5780162 , 0.57458603],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.5677104 ,
          0.56553656, 0.56327134],
         ...,
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.43449366,
          0.43854624, 0.451251  ],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.44263843,
          0.45143345, 0.46377662],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.46040422,
          0.4710226 , 0.48086703]],
 
        [[0.5026313 , 0.5026313 , 0.5026313 , ..., 0.5660189 ,
          0.57025343, 0.57178384],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.57619596,
          0.57681483, 0.5749774 ],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.5648095 ,
          0.5638342 , 0.56185734],
         ...,
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.43504322,
          0.43762857, 0.

In [6]:
def setFolder(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)

setFolder('../../../Model/Database/Target/images')
setFolder('../../../Model/Database/Target/masks')

for i, (img, mask) in enumerate(zip(images, masks)):
    np.save(f'../../../Model/Database/Target/images/img_{i:04d}.npy', img)
    np.save(f'../../../Model/Database/Target/masks/img_{i:04d}.npy', mask)